# Remote Sensing Super-Resolution — Complete Portable Project Companion

**40 original projects · nine stages · seven provenanced C/A/T families · six exact power-of-two scales.** This is a consolidated educational notebook, *not* an independently completed geographic research study. It preserves the original project IDs, does not modify the working Streamlit app, and retains canonical stage-specific experiments as the source of truth.

Read repository [AGENTS.md](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/AGENTS.md), [acceptance matrix](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/docs/acceptance_matrix.md), [completion brief](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/docs/completion_brief.md), and [resumability log](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/docs/next_session.md).

Run with Python 3.12 and offline base versions **numpy 2.3.5, Pillow 12.3.0, matplotlib 3.10.8, ipykernel 6.30.1, nbformat 5.10.4, nbclient 0.10.2**. Install in a separate environment. Optional GeoTIFF uses rasterio; optional trained inference uses a pinned source checkout plus PyTorch. No notebook cell performs installation, downloads, satellite lookup or training by default.

**Run All contract:** deterministic tiny synthetic RGB demo and all 40 chapter checks, without a mandatory network call. Model output is deliberately unavailable until authentic hash-verified weights are explicitly requested. The seven models are not renamed copies. A 100-step compatibility pilot is not held-out evaluation and high pixel density is not physical detail.

In [ ]:
import csv, hashlib, json, math, random, sys, platform
from pathlib import Path
import numpy as np
import PIL
from PIL import Image
from IPython.display import display
CONFIG = {
 "mode":"demo", "image":None,
 "bbox":[-114.0,43.0,-113.9,43.1], # WGS84 [west,south,east,north]
 "dates":["2026-10-01","2026-10-30"], "collection":"sentinel-2-l2a",
 "bands":["red","green","blue"], "cloud_threshold":20,
 "model_ids":["M1","M2","M3","M4","M5","M6","M7"],
 "scale":2, "aligned_HR":None, "device":"cpu", "seed":17,
 "memory_budget_mb":512, "download_budget_bytes":32000000,
 "max_output_pixels":1048576, "max_preview_side":256,
 "output_directory":"data/rssr_notebook_outputs",
 "run_models":False, "run_satellite":False, "run_training":False,
 "pinned_repo_path":None, "calibration_divisor":None,
}
assert CONFIG["mode"] in ("demo","upload","satellite")
assert CONFIG["scale"] in (2,4,8,16,32,64)
assert CONFIG["device"] in ("cpu","cuda")
np.random.seed(CONFIG["seed"]);random.seed(CONFIG["seed"])
print("Python",sys.version.split()[0],"NumPy",np.__version__,"Pillow",PIL.__version__, "Platform",platform.system())
print("Offline-first mode",CONFIG["mode"],"models explicitly enabled",CONFIG["run_models"])

## Exact-size arithmetic and resource protection

For source size \(W\times H\) and scale \(s\), output is exactly \(sW\times sH\), containing \(N=s^2WH\) pixels. RGB uint8 needs at least \(3N\) bytes; float32 RGB needs \(12N\), excluding activation and disk overhead. Model weights were originally trained for native ×2. For ×4/×8/×16/×32/×64, **cascade the native ×2 model** 2/3/4/5/6 times using the repository's disk-backed implementation. This is not six separately trained model scales. At 1024×1024 input and ×64 the output is 65,536×65,536 (4,294,967,296 pixels, ~12.9 GB uint8 RGB before overhead), so it is rejected by default.

In [ ]:
SCALES=(2,4,8,16,32,64)
def preflight(w,h,scale,max_pixels=1048576,memory_mb=512):
    if not all(type(k)==int for k in (w,h,scale)): raise TypeError("Exact integer geometry required")
    if not (1<=w<=1024 and 1<=h<=1024 and scale in SCALES): raise ValueError("Unsupported source size/scale")
    W,H=w*scale,h*scale; n=W*H
    rec={"output_width":W,"output_height":H,"output_pixels":n,
         "rgb_u8_bytes":3*n,"rgb_float32_bytes":12*n,"native2_passes":int(math.log2(scale)),
         "estimated_disk_bytes":sum(3*w*h*4**i for i in range(1,int(math.log2(scale))+1))+16*1024*1024}
    if n>max_pixels or 12*n>memory_mb*1024**2: raise MemoryError("Request exceeds configured output-pixel/memory budget; use a smaller AOI/scale or approved disk-backed job")
    return rec
assert preflight(5,3,64,max_pixels=1000000)["output_width"]==320
try: preflight(1024,1024,64)
except MemoryError: print("Correctly rejected 1024×1024 at ×64")
print("Default size:",preflight(16,16,CONFIG["scale"],CONFIG["max_output_pixels"],CONFIG["memory_budget_mb"]))

## Input validation and a known synthetic HR reference

The **default demo** creates 32×32 *synthetic* RGB ground truth, reduces it to 16×16 LR, and compares bicubic reconstruction against the known aligned fixture. For real RGB uploads, no HR metrics are reported until a genuinely matched, independently validated reference is supplied. GeoTIFF is kept in its native scientific bands with CRS/affine/nodata/units metadata and is never silently interpreted as an RGB photograph. A display stretch is not physical reflectance.

In [ ]:
from datetime import date
def checked_bbox(v):
    if len(v)!=4: raise ValueError("bbox must be [west,south,east,north]")
    a,b,c,d=map(float,v)
    if not all(map(math.isfinite,(a,b,c,d))) or not(-180<=a<c<=180 and -90<=b<d<=90):
        raise ValueError("Invalid WGS84 bbox")
    if c-a>1 or d-b>1: raise ValueError("Each bbox edge must be <=1 degree")
    return [a,b,c,d]
checked_bbox(CONFIG["bbox"])
assert date.fromisoformat(CONFIG["dates"][0])<=date.fromisoformat(CONFIG["dates"][1])
def open_source(config):
    mode=config["mode"]
    if mode=="demo":
        y,x=np.mgrid[:32,:32]
        hr=np.stack([x*8,y*8,(30+180*((x//4+y//4)%2))],-1).clip(0,255).astype("uint8")
        lr=np.asarray(Image.fromarray(hr).resize((16,16),Image.Resampling.BICUBIC))
        return lr,hr,{"kind":"synthetic RGB display","reference_valid":True}
    if mode=="upload":
        path=Path(config["image"] or "")
        if not path.is_file(): raise FileNotFoundError("Set image to an existing local PNG/JPEG/GeoTIFF")
        if path.suffix.lower() in (".png",".jpg",".jpeg"):
            with Image.open(path) as im: img=np.asarray(im.convert("RGB"))
            return img,None,{"kind":"display RGB","source":str(path),"reference_valid":False}
        if path.suffix.lower() in (".tif",".tiff"):
            import rasterio
            with rasterio.open(path) as ds:
                if ds.crs is None: raise ValueError("Scientific GeoTIFF CRS missing")
                if ds.width>1024 or ds.height>1024: raise ValueError("Source exceeds input limit")
                return None,None,{"kind":"scientific GeoTIFF; no RGB auto-conversion",
                    "source":str(path),"crs":str(ds.crs),"affine":list(ds.transform),
                    "count":ds.count,"descriptions":ds.descriptions,"units":ds.units,
                    "nodata":ds.nodata,"tags":ds.tags(),"reference_valid":False}
        raise ValueError("Supported inputs: PNG, JPEG, GeoTIFF")
    if not config["run_satellite"]: return None,None,{"kind":"satellite not requested","reference_valid":False}
    return None,None,{"kind":"satellite metadata opt-in","reference_valid":False}
lr,hr,source_record=open_source(CONFIG)
print("Source:",source_record)
if lr is not None:
    plan=preflight(lr.shape[1],lr.shape[0],CONFIG["scale"],CONFIG["max_output_pixels"],CONFIG["memory_budget_mb"])
    baseline=np.asarray(Image.fromarray(lr).resize((plan["output_width"],plan["output_height"]),Image.Resampling.BICUBIC))
    display(Image.fromarray(lr),Image.fromarray(baseline))
    print("LR",lr.shape,"bicubic",baseline.shape)

## Metric derivations and reference validity

For matched normalized arrays \(X,Y\in[0,1]\) over valid reference pixels \(\Omega\), \(\operatorname{MSE}=|\Omega|^{-1}C^{-1}\sum_{p,c}(X_{pc}-Y_{pc})^2\) and \(\mathrm{PSNR}=10\log_{10}(1/\mathrm{MSE})\). Example: normalized error 0.1 gives MSE 0.01 and PSNR 20 dB. SSIM compares luminance, contrast and covariance using positive stabilizers \(C_1,C_2\). The global numerical demonstration below is **not** window-averaged benchmark SSIM. Without valid co-registered HR truth return null, never a fabricated value. Real geospatial evaluation additionally requires aligned CRS/grid, masks, range, acquisition, sensor and geographic holdout.

In [ ]:
def metrics(pred,reference,mask=None):
    if reference is None: return {"status":"NO_VALID_ALIGNED_HR","MSE":None,"PSNR":None,"SSIM":None}
    if pred.shape!=reference.shape or pred.dtype!=np.uint8 or reference.dtype!=np.uint8:
        raise ValueError("Exactly aligned uint8 RGB pairs required")
    valid=np.ones(pred.shape[:2],bool) if mask is None else np.asarray(mask,dtype=bool)
    if valid.shape!=pred.shape[:2] or not valid.any(): raise ValueError("Empty/invalid reference mask")
    a=pred[valid].astype("float64")/255;b=reference[valid].astype("float64")/255
    mse=float(np.mean((a-b)**2))
    c1=.01**2;c2=.03**2
    ma,mb=a.mean(),b.mean();va,vb=a.var(),b.var()
    cov=np.mean((a-ma)*(b-mb))
    ssim=((2*ma*mb+c1)*(2*cov+c2))/((ma*ma+mb*mb+c1)*(va+vb+c2))
    return {"status":"SYNTHETIC_REFERENCE_ONLY" if CONFIG["mode"]=="demo" else "REFERENCE_REQUIRES_INDEPENDENT_REVIEW",
        "MSE":mse,"PSNR":float("inf") if mse==0 else 10*math.log10(1/mse),
        "SSIM_global_teaching_only":float(ssim),"valid_pixels":int(valid.sum())}
assert abs(10*math.log10(1/.01)-20)<1e-10
truth=hr if hr is not None and lr is not None and baseline.shape==hr.shape else None
metric_record=metrics(baseline,truth) if lr is not None else {"status":"NO_REFERENCE"}
print(json.dumps(metric_record,indent=2))

## Checkpoint registry (distinct trained paths)

All seven provenance records below are read directly from the repository's current `configs/hosted-models.json`. The models use genuine distinct checkpoint SHA-256 values; `architecture_id` differentiates C, A and T combinations. M2–M6 are 100-step RGB engineering pilots. M1/M7 1000-step versions have different training budgets. Comparing them does not establish a matched research ablation. Optional inference must use canonical `rssr.experiment.load_model` and `rssr.scaling.cascade`; no reimplementation or random-weight fallback is permitted.

In [ ]:
MODEL_CATALOG=json.loads("[{\"id\":\"M1\",\"architecture_id\":\"cat-v1/M1\",\"name\":\"CNN\",\"label\":\"RGB CNN v2 — 1000 training steps\",\"version\":\"b21b797a2c43f3a45879cd6778be4cd81600f2e7\",\"checkpoint\":{\"url\":\"https://raw.githubusercontent.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/b21b797a2c43f3a45879cd6778be4cd81600f2e7/artifacts/rgb-cnn-v2.pt\",\"sha256\":\"703b11deed506600f4f9504deb1e1fa662107db05086ce40be5179ae8cfab2af\",\"max_bytes\":262144},\"training_checkpoint_sha256\":\"b2091734910e077443cc51dd98014925d22f007d0604717953005a53280b5991\",\"limitations\":\"Frozen NAIP synthetic-x2 model evaluated on one held-out aerial scene. Real satellite generalization and physical calibration remain unvalidated. RGB CNN and hybrid were effectively tied; generated detail is not verified ground truth.\",\"source_license\":\"NAIP public domain; retain USDA/OpenSR attribution\"},{\"id\":\"M2\",\"architecture_id\":\"cat-v1/M2\",\"name\":\"Attention\",\"label\":\"M2 Attention RGB pilot — 100 training steps\",\"version\":\"v0.3.0-rgb-pilots\",\"checkpoint\":{\"url\":\"https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/releases/download/v0.3.0-rgb-pilots/m2-rgb-pilot-v1.pt\",\"sha256\":\"d9b819c4759e87fb8c2197f2d6fe775777be955944b1931bd8c60f50d7b07ccb\",\"max_bytes\":262144},\"training_checkpoint_sha256\":\"a25b60db652829259cf0dd247424c3146290bd2fe42591ee76e0dbcc0802ba8c\",\"limitations\":\"Genuine 100-step native2 RGB compatibility pilot, seed17 selected in advance. Reused training/validation tiles; no held-out evaluation. Not matched-budget or geographically validated research. Higher scales use cascades, not independently trained models.\",\"source_license\":\"NAIP public domain; retain USDA/OpenSR attribution\"},{\"id\":\"M3\",\"architecture_id\":\"cat-v1/M3\",\"name\":\"Transformer\",\"label\":\"M3 Transformer RGB pilot — 100 training steps\",\"version\":\"v0.3.0-rgb-pilots\",\"checkpoint\":{\"url\":\"https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/releases/download/v0.3.0-rgb-pilots/m3-rgb-pilot-v1.pt\",\"sha256\":\"cdd099b8bb292e3722585f8f341f510296956e1ed8306ea0e219b5a9c02852fc\",\"max_bytes\":262144},\"training_checkpoint_sha256\":\"0271e2b263216499056ce96c2abc56979798ed547c3fd286a13bf9255d00e52a\",\"limitations\":\"Genuine 100-step native2 RGB compatibility pilot, seed17 selected in advance. Reused training/validation tiles; no held-out evaluation. Not matched-budget or geographically validated research. Higher scales use cascades, not independently trained models.\",\"source_license\":\"NAIP public domain; retain USDA/OpenSR attribution\"},{\"id\":\"M4\",\"architecture_id\":\"cat-v1/M4\",\"name\":\"CNN + Attention\",\"label\":\"M4 CNN + Attention RGB pilot — 100 training steps\",\"version\":\"v0.3.0-rgb-pilots\",\"checkpoint\":{\"url\":\"https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/releases/download/v0.3.0-rgb-pilots/m4-rgb-pilot-v1.pt\",\"sha256\":\"0346073d5299ecb708c83a5cbe2e8f0d0f98430ee5c21466c74437bc56909940\",\"max_bytes\":262144},\"training_checkpoint_sha256\":\"a0d1746612f410abe067247e768e45a5d6e67fe06536056034152dad65969900\",\"limitations\":\"Genuine 100-step native2 RGB compatibility pilot, seed17 selected in advance. Reused training/validation tiles; no held-out evaluation. Not matched-budget or geographically validated research. Higher scales use cascades, not independently trained models.\",\"source_license\":\"NAIP public domain; retain USDA/OpenSR attribution\"},{\"id\":\"M5\",\"architecture_id\":\"cat-v1/M5\",\"name\":\"CNN + Transformer\",\"label\":\"M5 CNN + Transformer RGB pilot — 100 training steps\",\"version\":\"v0.3.0-rgb-pilots\",\"checkpoint\":{\"url\":\"https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/releases/download/v0.3.0-rgb-pilots/m5-rgb-pilot-v1.pt\",\"sha256\":\"dde37226d8b518826dbadc6801a64f7725d10bd43feef068058ee56bc0804c91\",\"max_bytes\":262144},\"training_checkpoint_sha256\":\"0d51af09289174402d55c89343e81f7e4cd62c72d58658b43e07b7673c0c1234\",\"limitations\":\"Genuine 100-step native2 RGB compatibility pilot, seed17 selected in advance. Reused training/validation tiles; no held-out evaluation. Not matched-budget or geographically validated research. Higher scales use cascades, not independently trained models.\",\"source_license\":\"NAIP public domain; retain USDA/OpenSR attribution\"},{\"id\":\"M6\",\"architecture_id\":\"cat-v1/M6\",\"name\":\"Attention + Transformer\",\"label\":\"M6 Attention + Transformer RGB pilot — 100 training steps\",\"version\":\"v0.3.0-rgb-pilots\",\"checkpoint\":{\"url\":\"https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/releases/download/v0.3.0-rgb-pilots/m6-rgb-pilot-v1.pt\",\"sha256\":\"c82403255c495e62c20776ef2158a2a25d88513d1f9c67ecb388c025b0cdf3d0\",\"max_bytes\":262144},\"training_checkpoint_sha256\":\"2bc27dcf4569eba8174198fadad3a3bca8150d875c082310a1a6112d375a6d17\",\"limitations\":\"Genuine 100-step native2 RGB compatibility pilot, seed17 selected in advance. Reused training/validation tiles; no held-out evaluation. Not matched-budget or geographically validated research. Higher scales use cascades, not independently trained models.\",\"source_license\":\"NAIP public domain; retain USDA/OpenSR attribution\"},{\"id\":\"M7\",\"architecture_id\":\"cat-v1/M7\",\"name\":\"CNN + Attention + Transformer\",\"label\":\"RGB CNN–attention–Transformer hybrid v2 — 1000 training steps\",\"version\":\"b21b797a2c43f3a45879cd6778be4cd81600f2e7\",\"checkpoint\":{\"url\":\"https://raw.githubusercontent.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/b21b797a2c43f3a45879cd6778be4cd81600f2e7/artifacts/rgb-hybrid-v2.pt\",\"sha256\":\"5cb8b9f45bc449bc6ff7ba4bf151a5ef1c70c0556d13c3b9f1c625995d7713f3\",\"max_bytes\":262144},\"training_checkpoint_sha256\":\"f6a64bb62d6209224be9fc924b57ef5c989f6869e4c7b925f7d1773bd2a780fc\",\"limitations\":\"Frozen NAIP synthetic-x2 model evaluated on one held-out aerial scene. Real satellite generalization and physical calibration remain unvalidated. RGB CNN and hybrid were effectively tied; generated detail is not verified ground truth.\",\"source_license\":\"NAIP public domain; retain USDA/OpenSR attribution\"}]")
assert len(MODEL_CATALOG)==7 and len({v["checkpoint"]["sha256"] for v in MODEL_CATALOG})==7
for v in MODEL_CATALOG: print(v["id"],v["name"],v["version"],v["checkpoint"]["sha256"][:20])
print("Seven real checkpoint identities; none downloaded or run by the default demo.")

## Optional authenticated-by-checksum model inference and disk-backed exports

Set `run_models=True` only after deliberately providing a **version-pinned canonical checkout** at `pinned_repo_path`. Install that checkout and its documented dependencies explicitly in your environment; do not copy unknown weights into the notebook. Cached paths are tied to SHA-256; unique run directories prevent overwriting. Canonical georeferenced TIFF processing is separate and must use `rssr.inference.geotiff` after radiometric and mask review.

In [ ]:
def run_canonical_models(image):
    if not CONFIG["run_models"]: return {},{"status":"not requested"}
    if image is None: raise ValueError("Scientific image must use calibrated canonical GeoTIFF pipeline, not RGB display")
    root=Path(CONFIG["pinned_repo_path"] or "").resolve()
    if not (root/"src/rssr/scaling.py").is_file(): raise FileNotFoundError("Set pinned_repo_path to the reviewed exact repository checkout")
    import rasterio
    import torch
    sys.path.insert(0,str(root/"src"))
    from rssr.hosted_assets import fetch_asset
    from rssr.experiment import load_model
    from rssr.scaling import cascade
    base=Path(CONFIG["output_directory"]);results={};failures={}
    for spec in MODEL_CATALOG:
        if spec["id"] not in CONFIG["model_ids"]:continue
        try:
            path=fetch_asset(spec["checkpoint"],base/"checkpoint_cache")
            digest=hashlib.sha256(path.read_bytes()).hexdigest()
            if digest!=spec["checkpoint"]["sha256"]:raise ValueError("Hash mismatch")
            model,state=load_model(path)
            if model.bands!=3 or model.scale!=2:raise ValueError("RGB native×2 incompatibility")
            target=base/("inference-"+spec["id"]+"-x"+str(CONFIG["scale"]))
            if target.exists():raise FileExistsError("Existing output directory; choose new output_directory")
            cascade(image,model,target,CONFIG["scale"],{"id":spec["id"],"checkpoint_sha256":digest})
            results[spec["id"]]={"output_directory":str(target),"checkpoint_sha256":digest,"status":"generated, not geographic accuracy evidence"}
        except Exception as exc:
            failures[spec["id"]]=type(exc).__name__+": "+str(exc)
    return results,failures
model_results,model_failures=run_canonical_models(lr)
print("Model runs:",model_results,"Model availability/errors:",model_failures)

## Optional bounded satellite metadata — official Element84 Earth Search

STAC `https://earth-search.aws.element84.com/v1` supports Sentinel-2 L2A discovery: https://element84.com/earth-search/examples/ . Search is explicitly opt-in, bbox ≤1° per edge, date range ≤366 days, ≤5 items, ≤2 MB metadata, and HTTP timeout 15 seconds. Timestamp means acquisition, *not* search time. Selected scene metadata includes ID, collection, processing level, cloud cover, and self URL. A catalogue result does not authorize raster download, establish scene-wide cloud-free quality or prove reflectance calibration.

To acquire pixels safely, use the canonical repository's bounded `rssr.fresh` path, check item/collection licences, per-band scale and offset, atmospheric level, SCL/QA masks, CRS/affine, nodata and band mapping. Never silently cast scientific reflectance to uint8 display RGB. This notebook defaults to no provider call.

In [ ]:
def stac_metadata(config):
    if not(config["mode"]=="satellite" and config["run_satellite"]):
        return {"status":"not requested; offline demo","provider":"Element84 Earth Search v1","items":[]}
    import urllib.request
    a,b=(date.fromisoformat(x) for x in config["dates"])
    if a>b or (b-a).days>366:raise ValueError("Invalid date interval")
    if config["collection"]!="sentinel-2-l2a":raise ValueError("Collection not reviewed")
    if not 0<=config["cloud_threshold"]<=100:raise ValueError("Cloud % out of range")
    payload={"bbox":checked_bbox(config["bbox"]),"collections":[config["collection"]],
      "datetime":f"{a}T00:00:00Z/{b}T23:59:59Z",
      "query":{"eo:cloud_cover":{"lt":config["cloud_threshold"]}},"limit":5}
    request=urllib.request.Request("https://earth-search.aws.element84.com/v1/search",
        json.dumps(payload).encode("utf-8"),{"Content-Type":"application/json"},method="POST")
    with urllib.request.urlopen(request,timeout=15) as resp: body=resp.read(min(2_000_000,config["download_budget_bytes"])+1)
    if len(body)>min(2_000_000,config["download_budget_bytes"]):raise ValueError("Metadata budget exceeded")
    items=json.loads(body)["features"][:5];out=[]
    for i in items:
        p=i.get("properties",{})
        out.append({"item_id":i.get("id"),"collection":i.get("collection"),"acquisition":p.get("datetime"),
           "created":p.get("created"),"cloud_pct":p.get("eo:cloud_cover"),
           "platform":p.get("platform"),"processing":p.get("processing:level"),
           "license":"VERIFY from authoritative STAC collection record","bands":list(i.get("assets",{})),
           "self_url":next((x.get("href") for x in i.get("links",[]) if x.get("rel")=="self"),None)})
    out.sort(key=lambda x:x["acquisition"] or "",reverse=True)
    return {"status":"bounded metadata only, not imagery acquisition","provider":"Element84 Earth Search v1","items":out}
satellite_provenance=stac_metadata(CONFIG)
print(json.dumps(satellite_provenance,indent=2)[:2400])

## Synchronized crops, preview and research-gated exports

The normalized crop center (u,v) is shared across comparison images, so resizing/cropping does not change the selected subject. Differences without HR are disagreements, **not ground-truth errors**. Export small default demonstrator only to ignored `data/` as PNG/CSV/JSON. For trained RGB/GeoTIFF use the canonical on-disk model pipeline. Check output dimensions before allocation.

In [ ]:
def crop(img,center=(.5,.5),zoom=2,max_side=256):
    a=np.asarray(img)
    if a.ndim!=3: raise ValueError("Expected an H×W×C image")
    h,w=a.shape[:2];size=max(1,min(int(min(h,w)/max(zoom,1)),max_side))
    cy,cx=int(center[1]*h),int(center[0]*w)
    y=max(0,min(h-size,cy-size//2));x=max(0,min(w-size,cx-size//2))
    return a[y:y+size,x:x+size]
if lr is not None:
    print("LR and bicubic synchronized preview shapes",crop(lr).shape,crop(baseline).shape)
    if CONFIG["mode"]=="demo":
        dest=Path(CONFIG["output_directory"]);dest.mkdir(parents=True,exist_ok=True)
        Image.fromarray(baseline).save(dest/"bicubic-teaching.png")
        with (dest/"metrics-teaching.csv").open("w",newline="",encoding="utf-8") as f:
            wr=csv.writer(f);wr.writerow(["measure","value"]);wr.writerows(metric_record.items())
        rec={"config_mode":CONFIG["mode"],"input":source_record,"reference":metric_record,
             "models":model_results,"unavailable":model_failures,"satellite":satellite_provenance,
             "interpretation":"Synthetic teaching fixture, not real satellite or trained-model evaluation"}
        (dest/"provenance-teaching.json").write_text(json.dumps(rec,indent=2,default=str),encoding="utf-8")
        print("Wrote PNG/CSV/JSON under",dest,"(do not commit outputs)")

## Interactive configuration

Plain Jupyter uses the editable `CONFIG` dictionary, which documents all required inputs. Optional ipywidgets (installed explicitly outside notebook) can collect mode, scale, model IDs, cloud threshold and an upload path, but do not automatically trigger downloads. Re-run selected cells after applying configuration. Scientific GeoTIFF is never converted implicitly.

In [ ]:
ENABLE_WIDGETS=False
if ENABLE_WIDGETS:
    import ipywidgets as widgets
    mode_ui=widgets.Dropdown(options=["demo","upload","satellite"],value=CONFIG["mode"],description="mode")
    scale_ui=widgets.Dropdown(options=SCALES,value=CONFIG["scale"],description="scale")
    models_ui=widgets.SelectMultiple(options=[m["id"] for m in MODEL_CATALOG],value=tuple(CONFIG["model_ids"]),description="models")
    image_ui=widgets.Text(value=CONFIG["image"] or "",description="image")
    cloud_ui=widgets.IntSlider(value=CONFIG["cloud_threshold"],min=0,max=100,description="cloud %")
    display(widgets.VBox([mode_ui,scale_ui,models_ui,image_ui,cloud_ui]))
    print("Explicitly update CONFIG from widget values and rerun; no hidden callbacks.")
else: print("Notebook works without widgets; configuration cell remains the canonical control surface.")

---

## MP01: Digital Image Explorer

**Stage:** stage_01_python_numpy. **Canonical chapter:** [MP01 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_01_python_numpy/MP01_digital_image_explorer/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_01_python_numpy/MP01_digital_image_explorer).

**Objective:** Pixels, RGB channels, grayscale images, image dimensions, bit depth, histograms.

**Mathematics / first-principles derivation:** Digital pixel at (row, col, channel) maps an integer in [0,255] to an RGB intensity; a H×W×C uint8 image occupies HWC bytes.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP01 — offline mathematical fixture (not a trained-model run)
print('MP01 image HWC and bytes:',(3,4,3),3*4*3)

---

## MP02: NumPy Image Laboratory

**Stage:** stage_01_python_numpy. **Canonical chapter:** [MP02 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_01_python_numpy/MP02_numpy_image_laboratory/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_01_python_numpy/MP02_numpy_image_laboratory).

**Objective:** Indexing, slicing, reshaping, broadcasting, normalization, vectorized operations.

**Mathematics / first-principles derivation:** Normalized pixels y=(x-min)/(max-min); guard constant arrays to avoid a zero denominator. Broadcasting applies one statistic per channel.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP02 — offline mathematical fixture (not a trained-model run)
print('MP02 normalization:',np.array([0,127.5,255])/255)

---

## MP03: Image Resizing Laboratory

**Stage:** stage_01_python_numpy. **Canonical chapter:** [MP03 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_01_python_numpy/MP03_image_resizing_laboratory/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_01_python_numpy/MP03_image_resizing_laboratory).

**Objective:** Nearest-neighbor, bilinear and bicubic interpolation.

**Mathematics / first-principles derivation:** Interpolation uses weighted neighboring samples: bilinear f(u,v)=Σ_{i,j∈{0,1}}w_ij f_ij; bicubic uses four positions per axis.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP03 — offline mathematical fixture (not a trained-model run)
print('MP03 bicubic output shape:',np.asarray(Image.fromarray(np.array([[0,255],[255,0]],dtype=np.uint8)).resize((4,4),Image.Resampling.BICUBIC)).shape)

---

## MP04: Image Comparison Tool

**Stage:** stage_01_python_numpy. **Canonical chapter:** [MP04 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_01_python_numpy/MP04_image_comparison_tool/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_01_python_numpy/MP04_image_comparison_tool).

**Objective:** Image differences, absolute error, MSE and visualization.

**Mathematics / first-principles derivation:** Absolute error D=|I-R|; MSE=(1/N)Σ(I-R)^2. Cast integer values before subtraction to avoid uint8 wraparound.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP04 — offline mathematical fixture (not a trained-model run)
print('MP04 MSE:',np.mean((np.array([0.,1.])-np.array([.2,.8]))**2))

---

## MP05: Gaussian Blur Simulator

**Stage:** stage_02_image_processing. **Canonical chapter:** [MP05 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_02_image_processing/MP05_gaussian_blur_simulator/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_02_image_processing/MP05_gaussian_blur_simulator).

**Objective:** Convolution, Gaussian kernels, kernel size and blur strength.

**Mathematics / first-principles derivation:** Gaussian G(x)=exp(-x²/(2σ²))/Z; larger σ spreads weight over a wider spatial neighborhood.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP05 — offline mathematical fixture (not a trained-model run)
q=np.arange(-2,3);g=np.exp(-q*q/2);g/=g.sum();print('MP05 Gaussian weights:',np.round(g,3))

---

## MP06: Low-Resolution Image Generator

**Stage:** stage_02_image_processing. **Canonical chapter:** [MP06 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_02_image_processing/MP06_low_resolution_image_generator/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_02_image_processing/MP06_low_resolution_image_generator).

**Objective:** Downsampling, scale factors, sampling and aliasing.

**Mathematics / first-principles derivation:** LR synthesis x_LR=D_s(h*x_HR), where h is antialiasing blur and D_s samples every s-th point.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP06 — offline mathematical fixture (not a trained-model run)
a=np.arange(64,dtype=np.uint8).reshape(8,8);print('MP06 LR shape:',np.asarray(Image.fromarray(a).resize((4,4))).shape)

---

## MP07: Synthetic Degradation Laboratory

**Stage:** stage_02_image_processing. **Canonical chapter:** [MP07 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_02_image_processing/MP07_synthetic_degradation_laboratory/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_02_image_processing/MP07_synthetic_degradation_laboratory).

**Objective:** Blur, noise, downsampling and degradation combinations.

**Mathematics / first-principles derivation:** Degradation model y=D_s(h*x)+ε; blur σ, noise variance and sampling scale each alter the inverse problem.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP07 — offline mathematical fixture (not a trained-model run)
rng=np.random.default_rng(7);print('MP07 noise MSE:',np.mean(rng.normal(0,.05,100)**2))

---

## MP08: Frequency-Domain Image Explorer

**Stage:** stage_02_image_processing. **Canonical chapter:** [MP08 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_02_image_processing/MP08_frequency_domain_image_explorer/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_02_image_processing/MP08_frequency_domain_image_explorer).

**Objective:** Fourier transforms, spatial frequencies, edge and texture information.

**Mathematics / first-principles derivation:** 2D DFT X[k,l]=Σx[m,n]exp[-j2π(km/M+ln/N)]; magnitude indicates Fourier energy distribution.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP08 — offline mathematical fixture (not a trained-model run)
print('MP08 FFT:',np.abs(np.fft.fft2(np.array([[1.,-1.],[1.,-1.]]))))

---

## MP09: MSE and PSNR from Scratch

**Stage:** stage_03_metrics. **Canonical chapter:** [MP09 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_03_metrics/MP09_mse_and_psnr_from_scratch/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_03_metrics/MP09_mse_and_psnr_from_scratch).

**Objective:** Reconstruction error, dynamic range and PSNR.

**Mathematics / first-principles derivation:** MSE and PSNR use a fixed normalized range L=1; a 0.1 error has MSE 0.01 and PSNR 20 dB.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP09 — offline mathematical fixture (not a trained-model run)
print('MP09 PSNR for error .1:',10*np.log10(1/.01))

---

## MP10: SSIM Evaluation Tool

**Stage:** stage_03_metrics. **Canonical chapter:** [MP10 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_03_metrics/MP10_ssim_evaluation_tool/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_03_metrics/MP10_ssim_evaluation_tool).

**Objective:** Luminance, contrast and structural similarity.

**Mathematics / first-principles derivation:** SSIM=((2μxμy+C1)(2σxy+C2))/((μx²+μy²+C1)(σx²+σy²+C2)); window protocol must be fixed.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP10 — offline mathematical fixture (not a trained-model run)
print('MP10 identical array global SSIM =',metrics(np.zeros((2,2,3),dtype=np.uint8),np.zeros((2,2,3),dtype=np.uint8))['SSIM_global_teaching_only'])

---

## MP11: Super-Resolution Evaluation Dashboard

**Stage:** stage_03_metrics. **Canonical chapter:** [MP11 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_03_metrics/MP11_super_resolution_evaluation_dashboard/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_03_metrics/MP11_super_resolution_evaluation_dashboard).

**Objective:** Metrics, error maps, comparison figures and result tables.

**Mathematics / first-principles derivation:** Report separate metrics, errors, valid reference masks and uncertainty; no single perceptual measure proves physical accuracy.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP11 — offline mathematical fixture (not a trained-model run)
print('MP11 reference gate:',metrics(np.zeros((1,1,3),dtype=np.uint8),None))

---

## MP12: Linear Algebra Visualizer

**Stage:** stage_04_math_pytorch. **Canonical chapter:** [MP12 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_04_math_pytorch/MP12_linear_algebra_visualizer/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_04_math_pytorch/MP12_linear_algebra_visualizer).

**Objective:** Vectors, matrices, tensor shapes, transformations and norms.

**Mathematics / first-principles derivation:** Matrix transform y=Ax, norm ||x||₂=(Σxᵢ²)½, covariance and eigenvectors describe variation in a feature space.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP12 — offline mathematical fixture (not a trained-model run)
print('MP12 matrix transform:',np.array([[2,0],[0,1]])@np.array([1,1]))

---

## MP13: Gradient Descent from Scratch

**Stage:** stage_04_math_pytorch. **Canonical chapter:** [MP13 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_04_math_pytorch/MP13_gradient_descent_from_scratch/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_04_math_pytorch/MP13_gradient_descent_from_scratch).

**Objective:** Derivatives, gradients, learning rates and optimization.

**Mathematics / first-principles derivation:** Gradient descent θ←θ−η∇L(θ); too-large η can diverge, too-small η converges slowly.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP13 — offline mathematical fixture (not a trained-model run)
x=3.
for _ in range(4): x-=.1*2*(x-1)
print('MP13 gradient descent x:',x)

---

## MP14: Neural Network from Scratch

**Stage:** stage_04_math_pytorch. **Canonical chapter:** [MP14 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_04_math_pytorch/MP14_neural_network_from_scratch/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_04_math_pytorch/MP14_neural_network_from_scratch).

**Objective:** Forward propagation, activation functions, loss and backpropagation.

**Mathematics / first-principles derivation:** Two-layer network h=ReLU(W₁x+b₁), y=W₂h+b₂; chain rule supplies backpropagation gradients.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP14 — offline mathematical fixture (not a trained-model run)
print('MP14 two-layer ReLU:',np.maximum(0,np.array([[.2,-.1],[.4,.3]])@np.array([1.,2.])))

---

## MP15: First PyTorch Neural Network

**Stage:** stage_04_math_pytorch. **Canonical chapter:** [MP15 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_04_math_pytorch/MP15_first_pytorch_neural_network/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_04_math_pytorch/MP15_first_pytorch_neural_network).

**Objective:** Tensors, autograd, optimizers and model training.

**Mathematics / first-principles derivation:** PyTorch autograd differentiates scalar loss through a tensor graph; compare its gradients with analytical differences.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP15 — offline mathematical fixture (not a trained-model run)
print('MP15 d(wx-y)^2/dw at w3,x2,y5:',2*(3*2-5)*2,'(PyTorch optional)')

---

## MP16: Reusable Training Pipeline

**Stage:** stage_04_math_pytorch. **Canonical chapter:** [MP16 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_04_math_pytorch/MP16_reusable_training_pipeline/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_04_math_pytorch/MP16_reusable_training_pipeline).

**Objective:** Dataset, DataLoader, validation, checkpoints and reproducibility.

**Mathematics / first-principles derivation:** Training pipeline separates fit/validation/held-out test; seed, optimizer, budget and checkpoint selection define the experiment.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP16 — offline mathematical fixture (not a trained-model run)
rng=np.random.default_rng(17);p=rng.permutation(12);print('MP16 split lengths:',len(p[:8]),len(p[8:10]),len(p[10:]))

---

## MP17: Convolution Playground

**Stage:** stage_05_cnn_sr. **Canonical chapter:** [MP17 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_05_cnn_sr/MP17_convolution_playground/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_05_cnn_sr/MP17_convolution_playground).

**Objective:** Kernels, padding, stride, feature maps and receptive fields.

**Mathematics / first-principles derivation:** Conv output at location is Σ_{i,j}K_{ij}X_{p+i,q+j}; stride and padding affect output size/receptive field.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP17 — offline mathematical fixture (not a trained-model run)
print('MP17 convolution mean:',np.mean(np.arange(9).reshape(3,3)))

---

## MP18: CNN Edge Detection Laboratory

**Stage:** stage_05_cnn_sr. **Canonical chapter:** [MP18 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_05_cnn_sr/MP18_cnn_edge_detection_laboratory/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_05_cnn_sr/MP18_cnn_edge_detection_laboratory).

**Objective:** Sobel filters and learned convolutional features.

**Mathematics / first-principles derivation:** Sobel kernels approximate horizontal/vertical image derivatives; edges respond to local contrast.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP18 — offline mathematical fixture (not a trained-model run)
print('MP18 Sobel response:',np.sum(np.array([[1,2,3]]*3)*np.array([[-1,0,1],[-2,0,2],[-1,0,1]])))

---

## MP19: CNN Image Reconstruction

**Stage:** stage_05_cnn_sr. **Canonical chapter:** [MP19 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_05_cnn_sr/MP19_cnn_image_reconstruction/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_05_cnn_sr/MP19_cnn_image_reconstruction).

**Objective:** Supervised reconstruction using convolutional neural networks.

**Mathematics / first-principles derivation:** CNN learns spatial filters with supervised MSE; toy residuals in this notebook are not trained predictions.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP19 — offline mathematical fixture (not a trained-model run)
print('MP19 toy residual:',np.array([.2,.4]) + np.array([.1,-.1]),'not trained')

---

## MP20: SRCNN Implementation

**Stage:** stage_05_cnn_sr. **Canonical chapter:** [MP20 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_05_cnn_sr/MP20_srcnn_implementation/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_05_cnn_sr/MP20_srcnn_implementation).

**Objective:** Feature extraction, nonlinear mapping and SR reconstruction.

**Mathematics / first-principles derivation:** SRCNN: bicubic upsample → convolutional feature extraction → nonlinear map → reconstruction; independent held-out table remains pending.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP20 — offline mathematical fixture (not a trained-model run)
print('MP20 SRCNN 3 convolution layers; baseline fixture MSE:',np.mean((np.array([.2,.4])-np.array([.3,.3]))**2))

---

## MP21: Residual CNN Super-Resolution

**Stage:** stage_05_cnn_sr. **Canonical chapter:** [MP21 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_05_cnn_sr/MP21_residual_cnn_super_resolution/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_05_cnn_sr/MP21_residual_cnn_super_resolution).

**Objective:** Residual blocks, skip connections and deep feature extraction.

**Mathematics / first-principles derivation:** Residual network y=x+F(x) learns a correction around a baseline; zero F exactly reproduces baseline.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP21 — offline mathematical fixture (not a trained-model run)
print('MP21 residual:',np.array([.3,.7])+np.array([-.1,.1]))

---

## MP22: Upsampling Architecture Comparison

**Stage:** stage_05_cnn_sr. **Canonical chapter:** [MP22 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_05_cnn_sr/MP22_upsampling_architecture_comparison/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_05_cnn_sr/MP22_upsampling_architecture_comparison).

**Objective:** Interpolation, transposed convolution and PixelShuffle.

**Mathematics / first-principles derivation:** PixelShuffle rearranges B·s² channels into B channels on a sH×sW grid without claiming new scene truth.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP22 — offline mathematical fixture (not a trained-model run)
print('MP22 PixelShuffle pre-channels for RGB x2:',3*4)

---

## MP23: Channel Attention from Scratch

**Stage:** stage_06_attention. **Canonical chapter:** [MP23 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_06_attention/MP23_channel_attention_from_scratch/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_06_attention/MP23_channel_attention_from_scratch).

**Objective:** Global pooling, channel weights and feature recalibration.

**Mathematics / first-principles derivation:** Channel attention uses global pooling and sigmoid gates: y_c=w_c x_c; weight is not calibrated uncertainty.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP23 — offline mathematical fixture (not a trained-model run)
print('MP23 channel gates:',1/(1+np.exp(-np.array([1.,2.]))))

---

## MP24: Spatial Attention from Scratch

**Stage:** stage_06_attention. **Canonical chapter:** [MP24 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_06_attention/MP24_spatial_attention_from_scratch/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_06_attention/MP24_spatial_attention_from_scratch).

**Objective:** Spatial attention maps and feature weighting.

**Mathematics / first-principles derivation:** Spatial attention multiplies each location by a learned mask after pooling features across channels.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP24 — offline mathematical fixture (not a trained-model run)
print('MP24 spatial gates:',1/(1+np.exp(-np.arange(4).reshape(2,2))))

---

## MP25: Combined Attention Module

**Stage:** stage_06_attention. **Canonical chapter:** [MP25 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_06_attention/MP25_combined_attention_module/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_06_attention/MP25_combined_attention_module).

**Objective:** Channel attention, spatial attention and CBAM-style integration.

**Mathematics / first-principles derivation:** CBAM style is y=M_s(M_c(F)⊙F)⊙(M_c(F)⊙F); distinguish channel from spatial modulation.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP25 — offline mathematical fixture (not a trained-model run)
print('MP25 CBAM toy shape:',(np.ones((2,2,2))*np.array([.3,.7])).shape)

---

## MP26: Attention-Based SR Network

**Stage:** stage_06_attention. **Canonical chapter:** [MP26 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_06_attention/MP26_attention_based_sr_network/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_06_attention/MP26_attention_based_sr_network).

**Objective:** Residual attention blocks and SR performance comparisons.

**Mathematics / first-principles derivation:** Attention SR combines learned residuals and attention modulation; compare to fixed bicubic on valid pairs.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP26 — offline mathematical fixture (not a trained-model run)
print('MP26 gated residual:',np.array([.3,.6])+.1/(1+np.exp(-np.array([.3,.6]))))

---

## MP27: Self-Attention from Scratch

**Stage:** stage_07_transformers. **Canonical chapter:** [MP27 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_07_transformers/MP27_self_attention_from_scratch/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_07_transformers/MP27_self_attention_from_scratch).

**Objective:** Queries, keys, values, scaled dot-product attention and softmax.

**Mathematics / first-principles derivation:** Scaled dot product attention softmax(QKᵀ/√dₖ)V; normalization is across keys for each query.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP27 — offline mathematical fixture (not a trained-model run)
q=np.eye(2);a=np.exp(q@q.T/np.sqrt(2));print('MP27 attention weights:',a/a.sum(1,keepdims=True))

---

## MP28: Multi-Head Self-Attention

**Stage:** stage_07_transformers. **Canonical chapter:** [MP28 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_07_transformers/MP28_multi_head_self_attention/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_07_transformers/MP28_multi_head_self_attention).

**Objective:** Attention heads, feature projections and concatenation.

**Mathematics / first-principles derivation:** Multi-head attention concatenates h independent attention projections before output projection.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP28 — offline mathematical fixture (not a trained-model run)
print('MP28 two head concatenation:',np.r_[np.array([1.,0.]),np.array([0.,1.])])

---

## MP29: Transformer Encoder Block

**Stage:** stage_07_transformers. **Canonical chapter:** [MP29 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_07_transformers/MP29_transformer_encoder_block/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_07_transformers/MP29_transformer_encoder_block).

**Objective:** Multi-head attention, normalization, residual connections and feed-forward networks.

**Mathematics / first-principles derivation:** Transformer block uses residual connections, normalization, self-attention and feed-forward layers.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP29 — offline mathematical fixture (not a trained-model run)
x=np.array([.1,.3,.9]);print('MP29 layer normalization:',(x-x.mean())/np.sqrt(x.var()+1e-5))

---

## MP30: Image Patches and Window Attention

**Stage:** stage_07_transformers. **Canonical chapter:** [MP30 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_07_transformers/MP30_image_patches_and_window_attention/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_07_transformers/MP30_image_patches_and_window_attention).

**Objective:** Patch embeddings, tokens, local windows and attention complexity.

**Mathematics / first-principles derivation:** Window attention costs O(N·w²) vs global O(N²) interactions for N tokens and w×w windows.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP30 — offline mathematical fixture (not a trained-model run)
print('MP30 8x8 global vs 4x4 window interactions:',64**2,64*16)

---

## MP31: Transformer-Based SR Baseline

**Stage:** stage_07_transformers. **Canonical chapter:** [MP31 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_07_transformers/MP31_transformer_based_sr_baseline/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_07_transformers/MP31_transformer_based_sr_baseline).

**Objective:** SwinIR-style image restoration and Transformer SR evaluation.

**Mathematics / first-principles derivation:** Transformer SR uses local attention and feed-forward residuals; trained SR requires verified compatible checkpoints.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP31 — offline mathematical fixture (not a trained-model run)
print('MP31 four 4x4 windows from 8x8; checkpoint required for SR:',(8//4)**2)

---

## MP32: Satellite Image Explorer

**Stage:** stage_08_remote_sensing. **Canonical chapter:** [MP32 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_08_remote_sensing/MP32_satellite_image_explorer/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_08_remote_sensing/MP32_satellite_image_explorer).

**Objective:** Satellite bands, spatial resolution, GeoTIFF and raster metadata.

**Mathematics / first-principles derivation:** Satellite raster stores channel count, CRS, affine, nodata and band semantics distinct from false-color RGB.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP32 — offline mathematical fixture (not a trained-model run)
print('MP32 satellite metadata fixture:',{'bands':['B02','B03','B04','B08'],'CRS':'EPSG:32643'})

---

## MP33: Satellite Preprocessing Pipeline

**Stage:** stage_08_remote_sensing. **Canonical chapter:** [MP33 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_08_remote_sensing/MP33_satellite_preprocessing_pipeline/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_08_remote_sensing/MP33_satellite_preprocessing_pipeline).

**Objective:** Band selection, normalization, invalid pixels and patch extraction.

**Mathematics / first-principles derivation:** Preprocessing must track raw DN, radiometric scale/offset, valid cloud/nodata masks, unit and train-fitted normalization.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP33 — offline mathematical fixture (not a trained-model run)
b=np.array([[1,0,2],[3,4,0]]);print('MP33 masked mean:',b[b>0].mean())

---

## MP34: Satellite HR–LR Pair Builder

**Stage:** stage_08_remote_sensing. **Canonical chapter:** [MP34 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_08_remote_sensing/MP34_satellite_hr_lr_pair_builder/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_08_remote_sensing/MP34_satellite_hr_lr_pair_builder).

**Objective:** Paired data, synthetic degradation and alignment.

**Mathematics / first-principles derivation:** Pair x_LR=D_s(x_HR) only for synthetic pairs; cross-sensor pairs need geometric and radiometric review.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP34 — offline mathematical fixture (not a trained-model run)
a=np.arange(64,dtype=np.uint8).reshape(8,8);print('MP34 synthetic x2 HR/LR:',a.shape,np.asarray(Image.fromarray(a).resize((4,4))).shape)

---

## MP35: Geographic Dataset Splitting

**Stage:** stage_08_remote_sensing. **Canonical chapter:** [MP35 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_08_remote_sensing/MP35_geographic_dataset_splitting/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_08_remote_sensing/MP35_geographic_dataset_splitting).

**Objective:** Scene-independent train, validation and test splits.

**Mathematics / first-principles derivation:** Split by region before tiling; disjoint geographic groups prevent near-identical neighbors leaking across splits.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP35 — offline mathematical fixture (not a trained-model run)
print('MP35 disjoint regions:',{'train':['A','B'],'val':['C'],'test':['D']})

---

## MP36: Cross-Sensor Dataset Investigation

**Stage:** stage_08_remote_sensing. **Canonical chapter:** [MP36 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_08_remote_sensing/MP36_cross_sensor_dataset_investigation/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_08_remote_sensing/MP36_cross_sensor_dataset_investigation).

**Objective:** WorldStrat, Sen2Venµs, geometric and radiometric differences.

**Mathematics / first-principles derivation:** Cross-sensor validity depends on viewing geometry, dates, calibration, point-spread function and spectral responses.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP36 — offline mathematical fixture (not a trained-model run)
print('MP36 sensor pixel size distinction (m):',10,20,'no cross-sensor HR implied')

---

## MP37: Feature Fusion Laboratory

**Stage:** stage_09_hybrid_research. **Canonical chapter:** [MP37 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_09_hybrid_research/MP37_feature_fusion_laboratory/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_09_hybrid_research/MP37_feature_fusion_laboratory).

**Objective:** CNN features, Transformer features, addition and concatenation.

**Mathematics / first-principles derivation:** Feature fusion z=C+T or z=W[C;T]; concatenation doubles channels before projection.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP37 — offline mathematical fixture (not a trained-model run)
c=np.array([1.,2.]);t=np.array([3.,4.]);print('MP37 fuse add/concat:',c+t,np.r_[c,t])

---

## MP38: Hybrid CNN–Attention–Transformer SR Model

**Stage:** stage_09_hybrid_research. **Canonical chapter:** [MP38 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_09_hybrid_research/MP38_hybrid_cnn_attention_transformer_sr_model/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_09_hybrid_research/MP38_hybrid_cnn_attention_transformer_sr_model).

**Objective:** Integrated architecture and reconstruction.

**Mathematics / first-principles derivation:** Hybrid shape trace: head H×W×C, fuse C/T branches, attention gate, PixelShuffle to 2H×2W×B.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP38 — offline mathematical fixture (not a trained-model run)
print('MP38 hybrid head/fuse/tail dimensions:',(1,8,5,7),(1,16,5,7),(1,3,10,14))

---

## MP39: Controlled Ablation Study

**Stage:** stage_09_hybrid_research. **Canonical chapter:** [MP39 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_09_hybrid_research/MP39_controlled_ablation_study/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_09_hybrid_research/MP39_controlled_ablation_study).

**Objective:** CNN, CNN + Attention, CNN + Transformer, full hybrid.

**Mathematics / first-principles derivation:** Ablations require equal splits, training budgets, evaluation protocol and checkpoint selection; names alone are no evidence.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP39 — offline mathematical fixture (not a trained-model run)
f=[(1,0,0),(0,1,0),(0,0,1),(1,1,0),(1,0,1),(0,1,1),(1,1,1)];assert len(set(f))==7;print('MP39 distinct flags:',f)

---

## MP40: Robustness and Efficiency Benchmark

**Stage:** stage_09_hybrid_research. **Canonical chapter:** [MP40 README](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/blob/main/stage_09_hybrid_research/MP40_robustness_and_efficiency_benchmark/README.md); [project notebook](https://github.com/Ahmdullah16/Mini-Project-ML-Lab-RSSR/tree/main/stage_09_hybrid_research/MP40_robustness_and_efficiency_benchmark).

**Objective:** Blur, noise, cross-sensor shift, parameters, FLOPs, inference time and GPU memory.

**Mathematics / first-principles derivation:** Robustness includes blur, noise, shift, sensor change; efficiency tracks latency, parameters, FLOPs, memory and tile seams.

**Controlled fixture:** the following executable cell computes a small actual numerical input/output, with printed values. Test parameters, boundary conditions and valid-domain assumptions before using real data. **Interpretation:** a numerical or shape sanity check confirms only this example; it does not prove the canonical experiment, measured trained performance or learner completion.

**Experiments, limitations, citations and completion gates:** refer to the linked canonical README, notebook and VALIDATION.md, which remain authoritative. Follow their documented data licence, prerequisite and evidence requirements.

```text
Input: explicit small deterministic vector / image / synthetic scene.
Output: measured numerical value, dimensions or explanation printed below.
```

In [ ]:
# MP40 — offline mathematical fixture (not a trained-model run)
r=np.random.default_rng(40).normal(0,.02,100);print('MP40 noise MSE fixture:',np.mean(r*r))

## Evidence, scientific validity and next steps

The forty offline calculations are teaching fixtures only; canonical notebooks remain independently maintained. Geographic holdout, genuine aligned HR, radiometric/atmospheric validation, multiband mask fidelity, matched-budget studies, real provider downloads, all-seven large-scale inference, and publication novelty are **not accepted** by running the default notebook.

Data and model checkpoints are explicitly versioned in the repository. Do not commit secrets, weights, provider data or large outputs. For a scientific GeoTIFF use canonical `rssr.inference.geotiff` with a verified calibration divisor and scientific band input. For large jobs use the bounded on-disk native ×2 stages; never allocate an unrestricted final array. Public Streamlit application and final submission documents remain separate deliverables.

In [ ]:
assert len(MODEL_CATALOG)==7
assert len({m["id"] for m in MODEL_CATALOG})==7
assert preflight(5,3,2)["output_width"]==10
assert metrics(np.zeros((1,1,3),dtype=np.uint8),None)["PSNR"] is None
print("Finished 40 deterministic teaching fixtures and core local invariants.")
print("External satellite, trained inference, geographic reference validation and training NOT run by default.")